In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from __future__ import annotations

import json
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple

import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from IPython.display import Image, display

import shap

warnings.filterwarnings("ignore")

In [ ]:
PROJECT_DIR = Path.cwd().parent.parent
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATE_COL = "date"
TARGET_REVENUE = "revenue"
TARGET_COGS = "cogs"


In [6]:


# =========================
# HELPERS
# =========================

def safe_div(a, b):
    return np.where((b == 0) | pd.isna(b), np.nan, a / b)

def load_csv(name, parse_dates=None):
    file_path = DATA_DIR / name

    if not file_path.exists():
        print(f"[WARN] Không tìm thấy {file_path}, bỏ qua.")
        return None

    return pd.read_csv(file_path, parse_dates=parse_dates, low_memory=False)

def pick_existing_cols(df, cols):
    if df is None:
        return []
    return [c for c in cols if c in df.columns]

def add_calendar_features(df, date_col=DATE_COL):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])

    df["day"] = df[date_col].dt.day
    df["day_of_week"] = df[date_col].dt.dayofweek
    df["day_of_year"] = df[date_col].dt.dayofyear
    df["week_of_year"] = df[date_col].dt.isocalendar().week.astype(int)
    df["month"] = df[date_col].dt.month
    df["quarter"] = df[date_col].dt.quarter
    df["year"] = df[date_col].dt.year

    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
    df["is_month_start"] = df[date_col].dt.is_month_start.astype(int)
    df["is_month_end"] = df[date_col].dt.is_month_end.astype(int)
    df["is_quarter_start"] = df[date_col].dt.is_quarter_start.astype(int)
    df["is_quarter_end"] = df[date_col].dt.is_quarter_end.astype(int)

    return df

def add_ratio_features(feat):
    feat = feat.copy()

    ratio_specs = [
        ("discount_per_order", "discount_total", "orders_cnt"),
        ("items_per_order", "items_qty_sum", "orders_cnt"),
        ("gmv_per_order", "gross_merch_value", "orders_cnt"),
        ("refund_per_order", "refund_amount_sum", "orders_cnt"),
        ("returns_per_order", "returns_cnt", "orders_cnt"),
        ("pages_per_session", "page_views_total", "sessions_total"),
        ("visitors_per_session", "unique_visitors_total", "sessions_total"),
    ]

    for new_col, num_col, den_col in ratio_specs:
        if {num_col, den_col}.issubset(feat.columns):
            feat[new_col] = safe_div(feat[num_col], feat[den_col])

    return feat

def add_cyclical_time_features(feat):
    feat = feat.copy()

    if "day_of_week" in feat.columns:
        feat["dow_sin"] = np.sin(2 * np.pi * feat["day_of_week"] / 7)
        feat["dow_cos"] = np.cos(2 * np.pi * feat["day_of_week"] / 7)

    if "month" in feat.columns:
        feat["month_sin"] = np.sin(2 * np.pi * feat["month"] / 12)
        feat["month_cos"] = np.cos(2 * np.pi * feat["month"] / 12)

    if "day_of_year" in feat.columns:
        feat["doy_sin"] = np.sin(2 * np.pi * feat["day_of_year"] / 365.25)
        feat["doy_cos"] = np.cos(2 * np.pi * feat["day_of_year"] / 365.25)

    return feat

def add_holiday_mega_sale_features(feat, date_col=DATE_COL):
    feat = feat.copy()
    feat[date_col] = pd.to_datetime(feat[date_col])

    mmdd = feat[date_col].dt.strftime("%m-%d")
    fixed_holidays = {
        "01-01",
        "04-30",
        "05-01",
        "09-02",
    }

    feat["is_holiday_fixed"] = mmdd.isin(fixed_holidays).astype(int)

    feat["is_mega_sale_0909"] = mmdd.eq("09-09").astype(int)
    feat["is_mega_sale_1111"] = mmdd.eq("11-11").astype(int)
    feat["is_mega_sale_1212"] = mmdd.eq("12-12").astype(int)

    feat["is_mega_sale"] = (
        feat["is_mega_sale_0909"]
        | feat["is_mega_sale_1111"]
        | feat["is_mega_sale_1212"]
    ).astype(int)

    mega_dates = feat.loc[feat["is_mega_sale"] == 1, date_col].tolist()
    if mega_dates:
        feat["days_to_next_mega_sale"] = np.nan
        feat["days_since_prev_mega_sale"] = np.nan

        for i, dt in enumerate(feat[date_col]):
            next_diffs = [(d - dt).days for d in mega_dates if d >= dt]
            prev_diffs = [(dt - d).days for d in mega_dates if d <= dt]

            feat.loc[i, "days_to_next_mega_sale"] = min(next_diffs) if next_diffs else np.nan
            feat.loc[i, "days_since_prev_mega_sale"] = min(prev_diffs) if prev_diffs else np.nan
    else:
        feat["days_to_next_mega_sale"] = np.nan
        feat["days_since_prev_mega_sale"] = np.nan

    feat["is_pre_mega_sale_3d"] = (
        feat["days_to_next_mega_sale"].between(1, 3, inclusive="both")
    ).astype(int)
    feat["is_post_mega_sale_3d"] = (
        feat["days_since_prev_mega_sale"].between(1, 3, inclusive="both")
    ).astype(int)

    return feat

def build_daily_promo_features(base_dates, promotions, date_col=DATE_COL):
    if promotions is None or len(promotions) == 0:
        return None

    promo = promotions.copy()
    promo["start_date"] = pd.to_datetime(promo["start_date"])
    promo["end_date"] = pd.to_datetime(promo["end_date"])

    promo["applicable_category"] = (
        promo["applicable_category"]
        .fillna("Any")
        .astype(str)
        .str.strip()
    )

    promo["promo_type"] = (
        promo["promo_type"]
        .fillna("unknown")
        .astype(str)
        .str.lower()
        .str.strip()
    )

    promo["discount_value"] = pd.to_numeric(
        promo["discount_value"], errors="coerce"
    ).fillna(0)

    if "stackable_flag" in promo.columns:
        promo["stackable_flag"] = pd.to_numeric(
            promo["stackable_flag"], errors="coerce"
        ).fillna(0)
    else:
        promo["stackable_flag"] = 0

    focus_categories = ["Outdoor", "Streetwear"]

    rows = []

    for dt in pd.to_datetime(base_dates[date_col]):
        active = promo[
            (promo["start_date"] <= dt) &
            (promo["end_date"] >= dt)
        ].copy()

        row = {date_col: dt}

        active_cnt = len(active)
        row["is_promo_active"] = int(active_cnt > 0)
        row["promo_active_cnt"] = active_cnt

        if active_cnt == 0:
            row["promo_global_cnt"] = 0
            row["promo_targeted_cnt"] = 0
            row["promo_global_ratio"] = 0
            row["promo_discount_mean"] = 0
            row["promo_global_discount_mean"] = 0
            row["promo_targeted_discount_mean"] = 0
            row["promo_intensity"] = 0
            row["promo_pct_cnt"] = 0
            row["promo_fixed_cnt"] = 0
            row["promo_stackable_cnt"] = 0

            for cat in focus_categories:
                key = cat.lower()
                row[f"promo_{key}_direct_cnt"] = 0
                row[f"promo_{key}_direct_discount_mean"] = 0

        else:
            global_mask = active["applicable_category"].eq("Any")
            targeted_mask = ~global_mask

            row["promo_global_cnt"] = int(global_mask.sum())
            row["promo_targeted_cnt"] = int(targeted_mask.sum())
            row["promo_global_ratio"] = row["promo_global_cnt"] / active_cnt

            row["promo_discount_mean"] = active["discount_value"].mean()

            row["promo_global_discount_mean"] = (
                active.loc[global_mask, "discount_value"].mean()
                if global_mask.any()
                else 0
            )

            row["promo_targeted_discount_mean"] = (
                active.loc[targeted_mask, "discount_value"].mean()
                if targeted_mask.any()
                else 0
            )

            row["promo_intensity"] = (
                row["promo_active_cnt"] * row["promo_discount_mean"]
            )

            row["promo_pct_cnt"] = int((active["promo_type"] == "percentage").sum())
            row["promo_fixed_cnt"] = int((active["promo_type"] == "fixed").sum())
            row["promo_stackable_cnt"] = int(active["stackable_flag"].sum())

            for cat in focus_categories:
                key = cat.lower()
                cat_mask = active["applicable_category"].eq(cat)

                row[f"promo_{key}_direct_cnt"] = int(cat_mask.sum())
                row[f"promo_{key}_direct_discount_mean"] = (
                    active.loc[cat_mask, "discount_value"].mean()
                    if cat_mask.any()
                    else 0
                )

        rows.append(row)

    out = pd.DataFrame(rows)

    # Window trước / sau promo
    promo_start_dates = [
    pd.Timestamp(x) for x in sorted(promo["start_date"].dropna().unique())
    ]

    promo_end_dates = [
        pd.Timestamp(x) for x in sorted(promo["end_date"].dropna().unique())
    ]

    def days_to_next(dt, dates):
        diffs = [(d - dt).days for d in dates if d >= dt]
        return min(diffs) if diffs else np.nan

    def days_since_prev(dt, dates):
        diffs = [(dt - d).days for d in dates if d <= dt]
        return min(diffs) if diffs else np.nan

    out["days_to_next_promo_start"] = [
        days_to_next(dt, promo_start_dates)
        for dt in out[date_col]
    ]

    out["days_since_prev_promo_end"] = [
        days_since_prev(dt, promo_end_dates)
        for dt in out[date_col]
    ]

    out["is_promo_start_day"] = (
        out["days_to_next_promo_start"] == 0
    ).astype(int)

    out["is_promo_end_day"] = (
        out["days_since_prev_promo_end"] == 0
    ).astype(int)

    out["is_pre_promo_3d"] = (
        out["days_to_next_promo_start"]
        .between(1, 3, inclusive="both")
    ).astype(int)

    out["is_post_promo_3d"] = (
        out["days_since_prev_promo_end"]
        .between(1, 3, inclusive="both")
    ).astype(int)

    out["is_pre_promo_7d"] = (
        out["days_to_next_promo_start"]
        .between(1, 7, inclusive="both")
    ).astype(int)

    out["is_post_promo_7d"] = (
        out["days_since_prev_promo_end"]
        .between(1, 7, inclusive="both")
    ).astype(int)

    return out

def add_lag_rolling_features(
    feat,
    cols,
    lags=(1, 7, 14, 28),
    wins=(7, 14, 28),
    spans=(7, 14, 28),
):
    feat = feat.copy()
    new_cols = []

    for col in cols:
        if col not in feat.columns:
            continue

        shifted = feat[col].shift(1)

        for lag in lags:
            new_cols.append(pd.Series(feat[col].shift(lag), name=f"{col}_lag_{lag}"))

        for win in wins:
            new_cols.append(pd.Series(shifted.rolling(win).mean(), name=f"{col}_roll_mean_{win}"))
            new_cols.append(pd.Series(shifted.rolling(win).std(), name=f"{col}_roll_std_{win}"))
            new_cols.append(pd.Series(shifted.rolling(win).sum(), name=f"{col}_roll_sum_{win}"))

        for span in spans:
            new_cols.append(
                pd.Series(
                    shifted.ewm(span=span, adjust=False).mean(),
                    name=f"{col}_ewm_mean_{span}",
                )
            )

    if new_cols:
        feat = pd.concat([feat] + new_cols, axis=1)

    return feat


# =========================
# LOAD DATA
# =========================

print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

products = load_csv("products.csv")
customers = load_csv("customers.csv", parse_dates=["signup_date"])
promotions = load_csv("promotions.csv", parse_dates=["start_date", "end_date"])
geography = load_csv("geography.csv")

orders = load_csv("orders.csv", parse_dates=["order_date"])
order_items = load_csv("order_items.csv")
payments = load_csv("payments.csv")
shipments = load_csv("shipments.csv", parse_dates=["ship_date", "delivery_date"])
returns = load_csv("returns.csv", parse_dates=["return_date"])
reviews = load_csv("reviews.csv", parse_dates=["review_date"])
sales = load_csv("sales.csv", parse_dates=["Date"])
inventory = load_csv("inventory.csv", parse_dates=["snapshot_date"])
web_traffic = load_csv("web_traffic.csv", parse_dates=["date"])
sample_submission = load_csv("sample_submission.csv", parse_dates=["Date"])

_ = promotions


# =========================
# BASE TABLE
# =========================

if sales is None:
    raise ValueError("Cần có data/sales.csv để làm bảng chính.")

sales = sales.rename(
    columns={
        "Date": DATE_COL,
        "Revenue": TARGET_REVENUE,
        "COGS": TARGET_COGS,
    }
)

if sample_submission is not None:
    future_dates = sample_submission.rename(columns={"Date": DATE_COL})[[DATE_COL]].copy()
    base_dates = (
        pd.concat([sales[[DATE_COL]], future_dates], ignore_index=True)
        .drop_duplicates()
        .sort_values(DATE_COL)
    )
else:
    base_dates = sales[[DATE_COL]].drop_duplicates().sort_values(DATE_COL).copy()

base = base_dates.merge(
    sales[[DATE_COL, TARGET_REVENUE, TARGET_COGS]],
    on=DATE_COL,
    how="left",
)
base = add_calendar_features(base, DATE_COL)


# =========================
# DAILY ORDERS
# =========================

daily_orders = None

if orders is not None:
    o = orders.copy()
    o[DATE_COL] = o["order_date"]

    daily_orders = (
        o.groupby(DATE_COL)
        .agg(
            orders_cnt=("order_id", "nunique"),
            unique_customers=("customer_id", "nunique"),
        )
        .reset_index()
    )

    pivot_specs = [
        ("order_status", "orders_status"),
        ("payment_method", "orders_paymethod"),
        ("device_type", "orders_device"),
        ("order_source", "orders_source"),
    ]

    for col, prefix in pivot_specs:
        if col in o.columns:
            tmp = pd.crosstab(o[DATE_COL], o[col])
            tmp.columns = [f"{prefix}_{c}" for c in tmp.columns]
            tmp = tmp.reset_index()
            daily_orders = daily_orders.merge(tmp, on=DATE_COL, how="left")


# =========================
# DAILY ORDER ITEMS
# =========================

daily_order_items = None

if order_items is not None and orders is not None:
    oi = order_items.copy()

    order_cols = pick_existing_cols(
        orders,
        ["order_id", "order_date", "customer_id", "zip", "order_status"],
    )
    oi = oi.merge(orders[order_cols], on="order_id", how="left")

    if products is not None and "product_id" in oi.columns and "product_id" in products.columns:
        product_cols = pick_existing_cols(
            products,
            ["product_id", "category", "segment", "size", "color", "price", "cogs"],
        )
        product_info = products[product_cols].copy()

        if "cogs" in product_info.columns:
            product_info = product_info.rename(columns={"cogs": "product_cogs"})

        oi = oi.merge(product_info, on="product_id", how="left")

    oi[DATE_COL] = oi["order_date"]

    if {"quantity", "unit_price"}.issubset(oi.columns):
        oi["gross_line_amount"] = oi["quantity"] * oi["unit_price"]
    else:
        oi["gross_line_amount"] = np.nan

    oi["promo_flag"] = oi["promo_id"].notna().astype(int) if "promo_id" in oi.columns else 0
    oi["promo2_flag"] = oi["promo_id_2"].notna().astype(int) if "promo_id_2" in oi.columns else 0

    if {"quantity", "product_cogs"}.issubset(oi.columns):
        oi["line_cogs_est"] = oi["quantity"] * oi["product_cogs"]
    else:
        oi["line_cogs_est"] = np.nan

    agg_dict = {
        "order_items_rows": ("order_id", "size"),
        "distinct_orders_from_items": ("order_id", "nunique"),
    }

    if "product_id" in oi.columns:
        agg_dict["distinct_products_sold"] = ("product_id", "nunique")
    if "quantity" in oi.columns:
        agg_dict["items_qty_sum"] = ("quantity", "sum")
    if "gross_line_amount" in oi.columns:
        agg_dict["gross_merch_value"] = ("gross_line_amount", "sum")
    if "discount_amount" in oi.columns:
        agg_dict["discount_total"] = ("discount_amount", "sum")
    if "promo_flag" in oi.columns:
        agg_dict["promo_item_cnt"] = ("promo_flag", "sum")
    if "promo2_flag" in oi.columns:
        agg_dict["promo2_item_cnt"] = ("promo2_flag", "sum")
    if "line_cogs_est" in oi.columns:
        agg_dict["est_cogs_from_items"] = ("line_cogs_est", "sum")

    daily_order_items = oi.groupby(DATE_COL).agg(**agg_dict).reset_index()

    if {"promo_item_cnt", "order_items_rows"}.issubset(daily_order_items.columns):
        daily_order_items["promo_item_ratio"] = safe_div(
            daily_order_items["promo_item_cnt"],
            daily_order_items["order_items_rows"],
        )

    if {"promo2_item_cnt", "order_items_rows"}.issubset(daily_order_items.columns):
        daily_order_items["promo2_item_ratio"] = safe_div(
            daily_order_items["promo2_item_cnt"],
            daily_order_items["order_items_rows"],
        )

    for col, prefix in [
        ("category", "cat_qty"),
        ("segment", "segment_qty"),
        ("size", "size_qty"),
    ]:
        if col in oi.columns and "quantity" in oi.columns:
            tmp = (
                oi.pivot_table(
                    index=DATE_COL,
                    columns=col,
                    values="quantity",
                    aggfunc="sum",
                    fill_value=0,
                )
                .add_prefix(f"{prefix}_")
                .reset_index()
            )
            daily_order_items = daily_order_items.merge(tmp, on=DATE_COL, how="left")


# =========================
# DAILY PAYMENTS
# =========================

daily_payments = None

if payments is not None and orders is not None:
    p = payments.copy()
    order_cols = pick_existing_cols(orders, ["order_id", "order_date"])

    p = p.merge(orders[order_cols], on="order_id", how="left")
    p[DATE_COL] = p["order_date"]

    agg_dict = {
        "payment_orders_cnt": ("order_id", "nunique"),
    }

    if "payment_value" in p.columns:
        agg_dict["payment_value_sum"] = ("payment_value", "sum")
        agg_dict["payment_value_mean"] = ("payment_value", "mean")

    if "installments" in p.columns:
        agg_dict["installments_mean"] = ("installments", "mean")
        agg_dict["installments_max"] = ("installments", "max")

    daily_payments = p.groupby(DATE_COL).agg(**agg_dict).reset_index()

    if "installments" in p.columns:
        tmp = pd.crosstab(p[DATE_COL], p["installments"])
        tmp.columns = [f"installments_{c}_cnt" for c in tmp.columns]
        tmp = tmp.reset_index()
        daily_payments = daily_payments.merge(tmp, on=DATE_COL, how="left")

    if "payment_method" in p.columns:
        tmp = pd.crosstab(p[DATE_COL], p["payment_method"])
        tmp.columns = [f"payments_method_{c}" for c in tmp.columns]
        tmp = tmp.reset_index()
        daily_payments = daily_payments.merge(tmp, on=DATE_COL, how="left")


# =========================
# DAILY SHIPMENTS
# =========================

daily_shipments = None

if shipments is not None:
    s = shipments.copy()

    if {"delivery_date", "ship_date"}.issubset(s.columns):
        s["delivery_days"] = (s["delivery_date"] - s["ship_date"]).dt.days
    else:
        s["delivery_days"] = np.nan

    s[DATE_COL] = s["ship_date"]

    agg_dict = {
        "shipments_cnt": ("order_id", "nunique"),
    }

    if "shipping_fee" in s.columns:
        agg_dict["shipping_fee_sum"] = ("shipping_fee", "sum")
        agg_dict["shipping_fee_mean"] = ("shipping_fee", "mean")

    if "delivery_days" in s.columns:
        agg_dict["delivery_days_mean"] = ("delivery_days", "mean")
        agg_dict["delivery_days_median"] = ("delivery_days", "median")

    daily_shipments = s.groupby(DATE_COL).agg(**agg_dict).reset_index()


# =========================
# DAILY RETURNS
# =========================

daily_returns = None

if returns is not None:
    r = returns.copy()
    r[DATE_COL] = r["return_date"]

    if products is not None and "product_id" in r.columns and "product_id" in products.columns:
        product_cols = pick_existing_cols(products, ["product_id", "category", "segment", "size"])
        r = r.merge(products[product_cols], on="product_id", how="left")

    agg_dict = {}

    if "return_id" in r.columns:
        agg_dict["returns_cnt"] = ("return_id", "nunique")
    if "return_quantity" in r.columns:
        agg_dict["return_qty_sum"] = ("return_quantity", "sum")
    if "refund_amount" in r.columns:
        agg_dict["refund_amount_sum"] = ("refund_amount", "sum")
    if "order_id" in r.columns:
        agg_dict["returned_orders_cnt"] = ("order_id", "nunique")
    if "product_id" in r.columns:
        agg_dict["returned_products_cnt"] = ("product_id", "nunique")

    daily_returns = r.groupby(DATE_COL).agg(**agg_dict).reset_index()

    if "return_reason" in r.columns:
        tmp = pd.crosstab(r[DATE_COL], r["return_reason"])
        tmp.columns = [f"return_reason_{c}" for c in tmp.columns]
        tmp = tmp.reset_index()
        daily_returns = daily_returns.merge(tmp, on=DATE_COL, how="left")


# =========================
# DAILY REVIEWS
# =========================

daily_reviews = None

if reviews is not None:
    rv = reviews.copy()
    rv[DATE_COL] = rv["review_date"]

    if "rating" in rv.columns:
        rv["low_rating_flag"] = (rv["rating"] <= 2).astype(int)
        rv["high_rating_flag"] = (rv["rating"] >= 4).astype(int)

    agg_dict = {}

    if "review_id" in rv.columns:
        agg_dict["reviews_cnt"] = ("review_id", "nunique")
    if "rating" in rv.columns:
        agg_dict["avg_rating"] = ("rating", "mean")
        agg_dict["median_rating"] = ("rating", "median")
        agg_dict["low_rating_cnt"] = ("low_rating_flag", "sum")
        agg_dict["high_rating_cnt"] = ("high_rating_flag", "sum")
    if "product_id" in rv.columns:
        agg_dict["unique_reviewed_products"] = ("product_id", "nunique")
    if "customer_id" in rv.columns:
        agg_dict["unique_review_customers"] = ("customer_id", "nunique")

    daily_reviews = rv.groupby(DATE_COL).agg(**agg_dict).reset_index()


# =========================
# DAILY WEB TRAFFIC
# =========================

daily_web_traffic = None

if web_traffic is not None:
    wt = web_traffic.copy()

    agg_dict = {}

    if "sessions" in wt.columns:
        agg_dict["sessions_total"] = ("sessions", "sum")
    if "unique_visitors" in wt.columns:
        agg_dict["unique_visitors_total"] = ("unique_visitors", "sum")
    if "page_views" in wt.columns:
        agg_dict["page_views_total"] = ("page_views", "sum")
    if "bounce_rate" in wt.columns:
        agg_dict["bounce_rate_mean"] = ("bounce_rate", "mean")
    if "avg_session_duration_sec" in wt.columns:
        agg_dict["avg_session_duration_sec_mean"] = ("avg_session_duration_sec", "mean")

    daily_web_traffic = wt.groupby(DATE_COL).agg(**agg_dict).reset_index()

    if "traffic_source" in wt.columns and "sessions" in wt.columns:
        source_sessions = (
            wt.pivot_table(
                index=DATE_COL,
                columns="traffic_source",
                values="sessions",
                aggfunc="sum",
                fill_value=0,
            )
            .add_prefix("sessions_source_")
            .reset_index()
        )
        daily_web_traffic = daily_web_traffic.merge(source_sessions, on=DATE_COL, how="left")

    if "traffic_source" in wt.columns and "unique_visitors" in wt.columns:
        source_visitors = (
            wt.pivot_table(
                index=DATE_COL,
                columns="traffic_source",
                values="unique_visitors",
                aggfunc="sum",
                fill_value=0,
            )
            .add_prefix("visitors_source_")
            .reset_index()
        )
        daily_web_traffic = daily_web_traffic.merge(source_visitors, on=DATE_COL, how="left")


# =========================
# DAILY CUSTOMER FEATURES
# =========================

daily_customers = None

if orders is not None and customers is not None:
    customer_cols = pick_existing_cols(
        customers,
        ["customer_id", "signup_date", "gender", "age_group", "acquisition_channel"],
    )

    oc = orders.merge(customers[customer_cols], on="customer_id", how="left")
    oc[DATE_COL] = oc["order_date"]

    if "signup_date" in oc.columns:
        oc["customer_tenure_days"] = (oc["order_date"] - oc["signup_date"]).dt.days
        oc["new_customer_same_day"] = (oc["customer_tenure_days"] == 0).astype(int)
    else:
        oc["customer_tenure_days"] = np.nan
        oc["new_customer_same_day"] = 0

    daily_customers = (
        oc.groupby(DATE_COL)
        .agg(
            unique_customers_from_profile=("customer_id", "nunique"),
            avg_customer_tenure_days=("customer_tenure_days", "mean"),
            median_customer_tenure_days=("customer_tenure_days", "median"),
            new_customers_same_day=("new_customer_same_day", "sum"),
        )
        .reset_index()
    )

    for col, prefix in [
        ("age_group", "age_group"),
        ("gender", "gender"),
    ]:
        if col in oc.columns:
            tmp = pd.crosstab(oc[DATE_COL], oc[col])
            tmp.columns = [f"{prefix}_{c}" for c in tmp.columns]
            tmp = tmp.reset_index()
            daily_customers = daily_customers.merge(tmp, on=DATE_COL, how="left")


# =========================
# DAILY GEOGRAPHY FEATURES
# =========================

daily_geography = None

if orders is not None and geography is not None:
    geo_cols = pick_existing_cols(geography, ["zip", "region", "city", "district"])
    og = orders.merge(geography[geo_cols], on="zip", how="left")
    og[DATE_COL] = og["order_date"]

    agg_dict = {}

    if "zip" in og.columns:
        agg_dict["shipping_zip_nunique"] = ("zip", "nunique")
    if "city" in og.columns:
        agg_dict["shipping_city_nunique"] = ("city", "nunique")
    if "district" in og.columns:
        agg_dict["shipping_district_nunique"] = ("district", "nunique")

    daily_geography = og.groupby(DATE_COL).agg(**agg_dict).reset_index()

    if "region" in og.columns:
        tmp = pd.crosstab(og[DATE_COL], og["region"])
        tmp.columns = [f"orders_region_{c}" for c in tmp.columns]
        tmp = tmp.reset_index()
        daily_geography = daily_geography.merge(tmp, on=DATE_COL, how="left")


# =========================
# INVENTORY DAILY PROXY
# =========================

daily_inventory = None

if inventory is not None:
    inv = inventory.copy()

    agg_dict = {}

    for col, agg_name in [
        ("stock_on_hand", "stock_on_hand_sum"),
        ("units_received", "units_received_sum"),
        ("units_sold", "units_sold_sum"),
        ("stockout_days", "stockout_days_sum"),
        ("stockout_flag", "stockout_flag_sum"),
        ("overstock_flag", "overstock_flag_sum"),
        ("reorder_flag", "reorder_flag_sum"),
    ]:
        if col in inv.columns:
            agg_dict[agg_name] = (col, "sum")

    for col, agg_name in [
        ("days_of_supply", "days_of_supply_mean"),
        ("fill_rate", "fill_rate_mean"),
        ("sell_through_rate", "sell_through_rate_mean"),
    ]:
        if col in inv.columns:
            agg_dict[agg_name] = (col, "mean")

    if "product_id" in inv.columns:
        agg_dict["product_inventory_nunique"] = ("product_id", "nunique")

    inv_month = inv.groupby("snapshot_date").agg(**agg_dict).reset_index()
    inv_month["apply_month"] = (
        inv_month["snapshot_date"] + pd.offsets.MonthBegin(1)
    ).dt.to_period("M")

    month_frame = pd.DataFrame({DATE_COL: base[DATE_COL].sort_values().unique()})
    month_frame["apply_month"] = month_frame[DATE_COL].dt.to_period("M")

    daily_inventory = (
        month_frame.merge(
            inv_month.drop(columns=["snapshot_date"]),
            on="apply_month",
            how="left",
        )
        .drop(columns=["apply_month"])
    )


# =========================
# MERGE DAILY TABLE
# =========================

daily_model_table = base.copy()

for df_part in [
    daily_orders,
    daily_order_items,
    daily_payments,
    daily_shipments,
    daily_returns,
    daily_reviews,
    daily_web_traffic,
    daily_customers,
    daily_geography,
    daily_inventory,
]:
    if df_part is not None:
        daily_model_table = daily_model_table.merge(df_part, on=DATE_COL, how="left")

daily_promo = build_daily_promo_features(base[[DATE_COL]].copy(), promotions, DATE_COL)

if daily_promo is not None:
    daily_model_table = daily_model_table.merge(daily_promo, on=DATE_COL, how="left")

# =========================
# BASIC DERIVED COLUMNS
# =========================

if {"gross_merch_value", "discount_total"}.issubset(daily_model_table.columns):
    daily_model_table["net_merch_after_discount"] = (
        daily_model_table["gross_merch_value"] - daily_model_table["discount_total"]
    )

if {"refund_amount_sum", "gross_merch_value"}.issubset(daily_model_table.columns):
    daily_model_table["refund_to_gmv_ratio"] = safe_div(
        daily_model_table["refund_amount_sum"],
        daily_model_table["gross_merch_value"],
    )

if {"returns_cnt", "orders_cnt"}.issubset(daily_model_table.columns):
    daily_model_table["return_per_order_ratio"] = safe_div(
        daily_model_table["returns_cnt"],
        daily_model_table["orders_cnt"],
    )

if {"promo_item_cnt", "items_qty_sum"}.issubset(daily_model_table.columns):
    daily_model_table["promo_item_to_qty_ratio"] = safe_div(
        daily_model_table["promo_item_cnt"],
        daily_model_table["items_qty_sum"],
    )

if {"page_views_total", "sessions_total"}.issubset(daily_model_table.columns):
    daily_model_table["pages_per_session"] = safe_div(
        daily_model_table["page_views_total"],
        daily_model_table["sessions_total"],
    )

if {"unique_visitors_total", "sessions_total"}.issubset(daily_model_table.columns):
    daily_model_table["visitors_per_session"] = safe_div(
        daily_model_table["unique_visitors_total"],
        daily_model_table["sessions_total"],
    )


# =========================
# SAVE RAW DAILY TABLE
# =========================

daily_model_table = daily_model_table.sort_values(DATE_COL).reset_index(drop=True)

raw_path = OUTPUT_DIR / "daily_model_table_raw.csv"
daily_model_table.to_csv(raw_path, index=False)

print("Saved raw daily table:", raw_path)
print("daily_model_table shape:", daily_model_table.shape)



# =========================
# CREATE FINAL FEATURE TABLE - TUNED BEST
# =========================

feat_best = daily_model_table[[DATE_COL, TARGET_REVENUE, TARGET_COGS]].copy()
feat_best = feat_best.sort_values(DATE_COL).reset_index(drop=True)

# calendar + cyclical
feat_best = add_calendar_features(feat_best, DATE_COL)
feat_best = add_cyclical_time_features(feat_best)
feat_best = add_holiday_mega_sale_features(feat_best, DATE_COL)

# trend features
feat_best["time_idx"] = (
    feat_best[DATE_COL] - feat_best[DATE_COL].min()
).dt.days

feat_best["time_idx_sq"] = feat_best["time_idx"] ** 2

# promo schedule features (known in advance)
promo_cols = [
    "is_promo_active",
    "promo_active_cnt",

    "promo_global_cnt",
    "promo_targeted_cnt",
    "promo_global_ratio",

    "promo_discount_mean",
    "promo_global_discount_mean",
    "promo_targeted_discount_mean",
    "promo_intensity",

    "promo_pct_cnt",
    "promo_fixed_cnt",
    "promo_stackable_cnt",

    "promo_outdoor_direct_cnt",
    "promo_outdoor_direct_discount_mean",

    "promo_streetwear_direct_cnt",
    "promo_streetwear_direct_discount_mean",

    "days_to_next_promo_start",
    "days_since_prev_promo_end",

    "is_promo_start_day",
    "is_promo_end_day",
    "is_pre_promo_3d",
    "is_post_promo_3d",
    "is_pre_promo_7d",
    "is_post_promo_7d",
]
promo_cols = [c for c in promo_cols if c in daily_model_table.columns]

if promo_cols:
    feat_best = feat_best.merge(
        daily_model_table[[DATE_COL] + promo_cols],
        on=DATE_COL,
        how="left",
    )

    distance_cols = [
        "days_to_next_promo_start",
        "days_since_prev_promo_end",
    ]
    distance_cols = [c for c in distance_cols if c in feat_best.columns]

    non_distance_promo_cols = [
        c for c in promo_cols
        if c not in distance_cols
    ]

    feat_best[non_distance_promo_cols] = (
        feat_best[non_distance_promo_cols].fillna(0)
    )

    feat_best[distance_cols] = (
        feat_best[distance_cols].fillna(999)
    )

# lag dài hơn cho target history
target_hist_cols = [c for c in [TARGET_REVENUE, TARGET_COGS] if c in feat_best.columns]
feat_best = add_lag_rolling_features(
    feat_best,
    target_hist_cols,
    lags=(1, 7, 14, 28, 56, 84, 182, 364, 365, 366),
    wins=(7, 14, 28, 56, 84),
    spans=(7, 14, 28, 56),
)

feat_best = feat_best.sort_values(DATE_COL).reset_index(drop=True)

best_features_path = OUTPUT_DIR / "daily_features_tuned_best.csv"
feat_best.to_csv(best_features_path, index=False)

print("Saved tuned-best feature table:", best_features_path)
print("daily_features_tuned_best shape:", feat_best.shape)



DATA_DIR: D:\CODEPY\Fashion_Ecom_Sales_Forecasting\data
OUTPUT_DIR: D:\CODEPY\Fashion_Ecom_Sales_Forecasting\outputs
Saved raw daily table: D:\CODEPY\Fashion_Ecom_Sales_Forecasting\outputs\daily_model_table_raw.csv
daily_model_table shape: (4381, 177)
Saved tuned-best feature table: D:\CODEPY\Fashion_Ecom_Sales_Forecasting\outputs\daily_features_tuned_best.csv
daily_features_tuned_best shape: (4381, 114)


In [7]:
PROJECT_DIR = Path("D:\CODEPY\Fashion_Ecom_Sales_Forecasting")
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
MODEL_DIR = OUTPUT_DIR / "models_deep_tune"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = OUTPUT_DIR / "daily_features_tuned_best.csv"
SAMPLE_SUBMISSION_FILE = DATA_DIR / "sample_submission.csv"

DATE_COL = "date"
TARGET_REVENUE = "revenue"
TARGET_COGS = "cogs"

RANDOM_SEED = 42
MAX_CPU_THREADS = os.cpu_count() or 4

N_FOLDS = 2
HORIZON = 60
MIN_TRAIN_ROWS = 730

LGBM_N_ITER = 12
XGB_N_ITER = 12
CAT_N_ITER = 12

SAVE_INTERMEDIATE = True
SAVE_MODELS = True
SAVE_ALT_SUBMISSIONS = False

if SAVE_MODELS:
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
# ============================================================
# OPTIONAL DEPENDENCIES
# ============================================================
try:
    import lightgbm as lgb
except Exception as exc:
    raise ImportError("Cần cài lightgbm trước: pip install lightgbm") from exc

try:
    from xgboost import XGBRegressor
except Exception as exc:
    raise ImportError("Cần cài xgboost trước: pip install xgboost") from exc

try:
    from catboost import CatBoostRegressor
except Exception as exc:
    raise ImportError("Cần cài catboost trước: pip install catboost") from exc


# ============================================================
# HELPERS
# ============================================================
def seed_everything(seed: int = 42) -> None:
    np.random.seed(seed)


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(mean_squared_error(y_true, y_pred) ** 0.5)


def eval_reg(y_true: np.ndarray, y_pred: np.ndarray, target_name: str) -> Dict[str, float]:
    return {
        "target": target_name,
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": rmse(y_true, y_pred),
        "r2": float(r2_score(y_true, y_pred)),
    }


def make_calendar_row(dt) -> Dict[str, float]:
    """
    Fallback row nếu known_future_df không có ngày tương lai.
    Bình thường script sẽ lấy known future feature trực tiếp từ daily_features_tuned_best.csv.
    """
    dt = pd.Timestamp(dt)
    mmdd = dt.strftime("%m-%d")

    row = {
        "day": dt.day,
        "day_of_week": dt.dayofweek,
        "day_of_year": dt.dayofyear,
        "week_of_year": int(dt.isocalendar().week),
        "month": dt.month,
        "quarter": dt.quarter,
        "year": dt.year,
        "is_weekend": int(dt.dayofweek >= 5),
        "is_month_start": int(dt.is_month_start),
        "is_month_end": int(dt.is_month_end),
        "is_quarter_start": int(dt.is_quarter_start),
        "is_quarter_end": int(dt.is_quarter_end),
        "dow_sin": np.sin(2 * np.pi * dt.dayofweek / 7),
        "dow_cos": np.cos(2 * np.pi * dt.dayofweek / 7),
        "month_sin": np.sin(2 * np.pi * dt.month / 12),
        "month_cos": np.cos(2 * np.pi * dt.month / 12),
        "doy_sin": np.sin(2 * np.pi * dt.dayofyear / 365.25),
        "doy_cos": np.cos(2 * np.pi * dt.dayofyear / 365.25),
        "is_holiday_fixed": int(mmdd in {"01-01", "04-30", "05-01", "09-02"}),
        "is_mega_sale_0909": int(mmdd == "09-09"),
        "is_mega_sale_1111": int(mmdd == "11-11"),
        "is_mega_sale_1212": int(mmdd == "12-12"),
    }

    row["is_mega_sale"] = int(
        row["is_mega_sale_0909"] or row["is_mega_sale_1111"] or row["is_mega_sale_1212"]
    )

    # fallback default cho known-future columns mới
    defaults = {
        "time_idx": np.nan,
        "time_idx_sq": np.nan,
        "days_to_next_mega_sale": 999,
        "days_since_prev_mega_sale": 999,
        "is_pre_mega_sale_3d": 0,
        "is_post_mega_sale_3d": 0,
        "is_promo_active": 0,
        "promo_active_cnt": 0,
        "promo_global_cnt": 0,
        "promo_targeted_cnt": 0,
        "promo_global_ratio": 0,
        "promo_discount_mean": 0,
        "promo_global_discount_mean": 0,
        "promo_targeted_discount_mean": 0,
        "promo_intensity": 0,
        "promo_pct_cnt": 0,
        "promo_fixed_cnt": 0,
        "promo_stackable_cnt": 0,
        "promo_outdoor_direct_cnt": 0,
        "promo_outdoor_direct_discount_mean": 0,
        "promo_streetwear_direct_cnt": 0,
        "promo_streetwear_direct_discount_mean": 0,
        "days_to_next_promo_start": 999,
        "days_since_prev_promo_end": 999,
        "is_promo_start_day": 0,
        "is_promo_end_day": 0,
        "is_pre_promo_3d": 0,
        "is_post_promo_3d": 0,
        "is_pre_promo_7d": 0,
        "is_post_promo_7d": 0,
    }
    row.update(defaults)
    return row


def make_ts_features(history_series: pd.Series, prefix: str, feature_cols: List[str]) -> Dict[str, float]:
    feats: Dict[str, float] = {}
    s = history_series.sort_index()

    for col in feature_cols:
        if col.startswith(f"{prefix}_lag_"):
            lag = int(col.split(f"{prefix}_lag_")[1])
            feats[col] = s.iloc[-lag] if len(s) >= lag else np.nan

        elif col.startswith(f"{prefix}_roll_mean_"):
            win = int(col.split(f"{prefix}_roll_mean_")[1])
            feats[col] = s.iloc[-win:].mean() if len(s) >= win else np.nan

        elif col.startswith(f"{prefix}_roll_std_"):
            win = int(col.split(f"{prefix}_roll_std_")[1])
            feats[col] = s.iloc[-win:].std(ddof=1) if len(s) >= win else np.nan

        elif col.startswith(f"{prefix}_roll_sum_"):
            win = int(col.split(f"{prefix}_roll_sum_")[1])
            feats[col] = s.iloc[-win:].sum() if len(s) >= win else np.nan

        elif col.startswith(f"{prefix}_ewm_mean_"):
            span = int(col.split(f"{prefix}_ewm_mean_")[1])
            feats[col] = s.ewm(span=span, adjust=False).mean().iloc[-1] if len(s) > 0 else np.nan

    return feats


def build_backtest_splits(
    df: pd.DataFrame,
    n_folds: int,
    horizon: int,
    min_train_rows: int,
) -> List[Tuple[np.ndarray, np.ndarray]]:
    n = len(df)
    splits: List[Tuple[np.ndarray, np.ndarray]] = []

    for i in range(n_folds, 0, -1):
        valid_start = n - i * horizon
        valid_end = valid_start + horizon
        if valid_start < min_train_rows or valid_end > n:
            continue
        train_idx = np.arange(0, valid_start)
        valid_idx = np.arange(valid_start, valid_end)
        splits.append((train_idx, valid_idx))

    return splits


@dataclass
class RuntimeChoice:
    requested: str
    actual: str


def _try_fit_dummy(model) -> bool:
    x = pd.DataFrame({"a": np.random.randn(64), "b": np.random.randn(64)})
    y = np.random.randn(64)
    try:
        model.fit(x, y)
        return True
    except Exception:
        return False


def detect_runtime_choices() -> Dict[str, RuntimeChoice]:
    choices: Dict[str, RuntimeChoice] = {}

    try:
        gpu_model = lgb.LGBMRegressor(
            objective="regression",
            n_estimators=20,
            learning_rate=0.1,
            num_leaves=31,
            random_state=RANDOM_SEED,
            n_jobs=MAX_CPU_THREADS,
            device_type="gpu",
            verbosity=-1,
        )
        choices["lgbm"] = RuntimeChoice("gpu", "gpu" if _try_fit_dummy(gpu_model) else "cpu")
    except Exception:
        choices["lgbm"] = RuntimeChoice("gpu", "cpu")

    try:
        gpu_model = XGBRegressor(
            n_estimators=20,
            max_depth=4,
            learning_rate=0.1,
            tree_method="hist",
            device="cuda",
            random_state=RANDOM_SEED,
            n_jobs=MAX_CPU_THREADS,
            objective="reg:squarederror",
            verbosity=0,
        )
        choices["xgb"] = RuntimeChoice("gpu", "gpu" if _try_fit_dummy(gpu_model) else "cpu")
    except Exception:
        choices["xgb"] = RuntimeChoice("gpu", "cpu")

    try:
        gpu_model = CatBoostRegressor(
            iterations=20,
            depth=4,
            learning_rate=0.1,
            loss_function="RMSE",
            task_type="GPU",
            random_seed=RANDOM_SEED,
            verbose=False,
        )
        choices["cat"] = RuntimeChoice("gpu", "gpu" if _try_fit_dummy(gpu_model) else "cpu")
    except Exception:
        choices["cat"] = RuntimeChoice("gpu", "cpu")

    return choices


def build_model(family: str, params: Dict, runtime_choices: Dict[str, RuntimeChoice]):
    runtime = runtime_choices[family].actual

    if family == "lgbm":
        base = dict(
            objective="regression",
            random_state=RANDOM_SEED,
            n_jobs=MAX_CPU_THREADS,
            verbosity=-1,
        )
        if runtime == "gpu":
            base["device_type"] = "gpu"
        return lgb.LGBMRegressor(**base, **params)

    if family == "xgb":
        base = dict(
            objective="reg:squarederror",
            random_state=RANDOM_SEED,
            n_jobs=MAX_CPU_THREADS,
            verbosity=0,
            tree_method="hist",
        )
        if runtime == "gpu":
            base["device"] = "cuda"
        return XGBRegressor(**base, **params)

    if family == "cat":
        base = dict(
            loss_function="RMSE",
            random_seed=RANDOM_SEED,
            thread_count=MAX_CPU_THREADS,
            verbose=False,
        )
        if runtime == "gpu":
            base["task_type"] = "GPU"
        return CatBoostRegressor(**base, **params)

    raise ValueError(f"Unsupported family: {family}")


def sample_lgbm_params(rng: np.random.Generator) -> Dict:
    return {
        "n_estimators": int(rng.choice([400, 600, 800, 1000, 1200])),
        "learning_rate": float(rng.choice([0.02, 0.03, 0.05, 0.08])),
        "num_leaves": int(rng.choice([31, 63, 95, 127])),
        "max_depth": int(rng.choice([-1, 6, 8, 10])),
        "min_child_samples": int(rng.choice([20, 40, 60, 80, 120])),
        "subsample": float(rng.choice([0.75, 0.85, 0.95, 1.0])),
        "colsample_bytree": float(rng.choice([0.7, 0.8, 0.9, 1.0])),
        "reg_alpha": float(rng.choice([0.0, 0.01, 0.05, 0.1, 0.5])),
        "reg_lambda": float(rng.choice([0.0, 0.01, 0.05, 0.1, 0.5, 1.0])),
    }


def sample_xgb_params(rng: np.random.Generator) -> Dict:
    return {
        "n_estimators": int(rng.choice([300, 500, 800, 1000])),
        "learning_rate": float(rng.choice([0.03, 0.05, 0.08])),
        "max_depth": int(rng.choice([3, 4, 6, 8])),
        "min_child_weight": float(rng.choice([1, 2, 4, 8])),
        "subsample": float(rng.choice([0.75, 0.85, 0.95, 1.0])),
        "colsample_bytree": float(rng.choice([0.7, 0.8, 0.9, 1.0])),
        "reg_alpha": float(rng.choice([0.0, 0.01, 0.05, 0.1])),
        "reg_lambda": float(rng.choice([0.5, 1.0, 1.5, 2.0])),
    }


def sample_cat_params(rng: np.random.Generator) -> Dict:
    return {
        "iterations": int(rng.choice([400, 700, 1000])),
        "learning_rate": float(rng.choice([0.03, 0.05, 0.08])),
        "depth": int(rng.choice([4, 6, 8])),
        "l2_leaf_reg": float(rng.choice([1, 3, 5, 7, 9])),
        "random_strength": float(rng.choice([0.0, 0.5, 1.0])),
    }


def sample_params(family: str, rng: np.random.Generator) -> Dict:
    if family == "lgbm":
        return sample_lgbm_params(rng)
    if family == "xgb":
        return sample_xgb_params(rng)
    if family == "cat":
        return sample_cat_params(rng)
    raise ValueError(f"Unsupported family: {family}")


def recursive_forecast(
    model,
    train_df: pd.DataFrame,
    future_dates: List[pd.Timestamp],
    target_col: str,
    feature_cols: List[str],
    known_future_df: pd.DataFrame | None = None,
) -> pd.DataFrame:
    history_series = train_df.set_index(DATE_COL)[target_col].copy().sort_index()
    feature_medians = train_df[feature_cols].median(numeric_only=True).to_dict()
    preds: List[float] = []

    known_future_map = None
    if known_future_df is not None:
        known_future_map = known_future_df.copy()
        known_future_map[DATE_COL] = pd.to_datetime(known_future_map[DATE_COL])
        known_future_map = known_future_map.set_index(DATE_COL)

    for dt in future_dates:
        dt = pd.Timestamp(dt)

        if known_future_map is not None and dt in known_future_map.index:
            base_row = known_future_map.loc[dt].to_dict()
            row = {k: base_row.get(k, np.nan) for k in feature_cols}
        else:
            row = make_calendar_row(dt)

        # Target-history features phải cập nhật theo recursive prediction
        row.update(make_ts_features(history_series, target_col, feature_cols))

        x_row = pd.DataFrame([row])
        for col in feature_cols:
            if col not in x_row.columns:
                x_row[col] = np.nan

        x_row = x_row[feature_cols].fillna(feature_medians).fillna(0.0)

        pred = float(model.predict(x_row)[0])
        pred = max(pred, 0.0)
        preds.append(pred)
        history_series.loc[dt] = pred

    return pd.DataFrame({DATE_COL: future_dates, target_col: preds})


def evaluate_family_params_recursive(
    df_target: pd.DataFrame,
    target_col: str,
    feature_cols: List[str],
    family: str,
    params: Dict,
    splits: List[Tuple[np.ndarray, np.ndarray]],
    runtime_choices: Dict[str, RuntimeChoice],
) -> Tuple[Dict, List[Dict]]:
    fold_rows: List[Dict] = []

    for fold_id, (tr_idx, va_idx) in enumerate(splits, start=1):
        train_df = df_target.iloc[tr_idx].copy().reset_index(drop=True)
        valid_df = df_target.iloc[va_idx].copy().reset_index(drop=True)

        model = build_model(family, params, runtime_choices)
        model.fit(train_df[feature_cols], train_df[target_col])

        pred_df = recursive_forecast(
            model=model,
            train_df=train_df[[DATE_COL, target_col] + feature_cols],
            future_dates=valid_df[DATE_COL].tolist(),
            target_col=target_col,
            feature_cols=feature_cols,
            known_future_df=valid_df[[DATE_COL] + feature_cols],
        )

        merged = valid_df[[DATE_COL, target_col]].merge(
            pred_df,
            on=DATE_COL,
            how="left",
            suffixes=("_true", "_pred"),
        )
        y_true = merged[f"{target_col}_true"].values
        y_pred = merged[f"{target_col}_pred"].values

        scores = eval_reg(y_true, y_pred, target_col)
        fold_rows.append(
            {
                "fold": fold_id,
                "family": family,
                "runtime_used": runtime_choices[family].actual,
                "valid_start": str(valid_df[DATE_COL].min().date()),
                "valid_end": str(valid_df[DATE_COL].max().date()),
                **scores,
            }
        )

    fold_df = pd.DataFrame(fold_rows)
    summary = {
        "target": target_col,
        "model_family": family,
        "mean_mae": float(fold_df["mae"].mean()),
        "std_mae": float(fold_df["mae"].std(ddof=0)),
        "mean_rmse": float(fold_df["rmse"].mean()),
        "std_rmse": float(fold_df["rmse"].std(ddof=0)),
        "mean_r2": float(fold_df["r2"].mean()),
        "std_r2": float(fold_df["r2"].std(ddof=0)),
        "folds": int(len(fold_df)),
        "runtime_used": runtime_choices[family].actual,
        "params_json": json.dumps(params, ensure_ascii=False, sort_keys=True),
    }
    return summary, fold_rows


def random_search_family(
    df_target: pd.DataFrame,
    target_col: str,
    feature_cols: List[str],
    family: str,
    n_iter: int,
    splits: List[Tuple[np.ndarray, np.ndarray]],
    runtime_choices: Dict[str, RuntimeChoice],
    seed_offset: int = 0,
) -> Tuple[pd.DataFrame, Dict, pd.DataFrame]:
    rng = np.random.default_rng(RANDOM_SEED + seed_offset)
    trial_rows: List[Dict] = []
    best_summary = None
    best_fold_rows: List[Dict] = []

    for trial in range(1, n_iter + 1):
        params = sample_params(family, rng)
        summary, fold_rows = evaluate_family_params_recursive(
            df_target=df_target,
            target_col=target_col,
            feature_cols=feature_cols,
            family=family,
            params=params,
            splits=splits,
            runtime_choices=runtime_choices,
        )
        summary["trial"] = trial
        trial_rows.append(summary)

        if best_summary is None or summary["mean_rmse"] < best_summary["mean_rmse"]:
            best_summary = summary.copy()
            best_fold_rows = fold_rows.copy()

        print(
            f"[{target_col}] {family} trial {trial}/{n_iter} | "
            f"RMSE={summary['mean_rmse']:.3f} | "
            f"MAE={summary['mean_mae']:.3f} | "
            f"R2={summary['mean_r2']:.4f}"
        )

    trials_df = pd.DataFrame(trial_rows).sort_values(["mean_rmse", "mean_mae"]).reset_index(drop=True)
    best_folds_df = pd.DataFrame(best_fold_rows)
    assert best_summary is not None
    return trials_df, best_summary, best_folds_df


def fit_full_model(
    df_target: pd.DataFrame,
    target_col: str,
    feature_cols: List[str],
    family: str,
    params: Dict,
    runtime_choices: Dict[str, RuntimeChoice],
):
    model = build_model(family, params, runtime_choices)
    model.fit(df_target[feature_cols], df_target[target_col])
    return model


def evaluate_ensemble_best_models(
    df_target: pd.DataFrame,
    target_col: str,
    feature_cols: List[str],
    best_params_by_family: Dict[str, Dict],
    splits: List[Tuple[np.ndarray, np.ndarray]],
    runtime_choices: Dict[str, RuntimeChoice],
) -> Tuple[Dict, pd.DataFrame]:
    fold_rows: List[Dict] = []

    for fold_id, (tr_idx, va_idx) in enumerate(splits, start=1):
        train_df = df_target.iloc[tr_idx].copy().reset_index(drop=True)
        valid_df = df_target.iloc[va_idx].copy().reset_index(drop=True)
        pred_stack: List[np.ndarray] = []

        for family, params in best_params_by_family.items():
            model = build_model(family, params, runtime_choices)
            model.fit(train_df[feature_cols], train_df[target_col])
            pred_df = recursive_forecast(
                model=model,
                train_df=train_df[[DATE_COL, target_col] + feature_cols],
                future_dates=valid_df[DATE_COL].tolist(),
                target_col=target_col,
                feature_cols=feature_cols,
                known_future_df=valid_df[[DATE_COL] + feature_cols],
            )
            pred_stack.append(pred_df[target_col].values)

        ensemble_pred = np.mean(np.vstack(pred_stack), axis=0)
        y_true = valid_df[target_col].values
        scores = eval_reg(y_true, ensemble_pred, target_col)
        fold_rows.append(
            {
                "fold": fold_id,
                "family": "ensemble_mean",
                "valid_start": str(valid_df[DATE_COL].min().date()),
                "valid_end": str(valid_df[DATE_COL].max().date()),
                **scores,
            }
        )

    fold_df = pd.DataFrame(fold_rows)
    summary = {
        "target": target_col,
        "model_family": "ensemble_mean",
        "mean_mae": float(fold_df["mae"].mean()),
        "std_mae": float(fold_df["mae"].std(ddof=0)),
        "mean_rmse": float(fold_df["rmse"].mean()),
        "std_rmse": float(fold_df["rmse"].std(ddof=0)),
        "mean_r2": float(fold_df["r2"].mean()),
        "std_r2": float(fold_df["r2"].std(ddof=0)),
        "folds": int(len(fold_df)),
        "runtime_used": "mixed",
        "params_json": json.dumps(best_params_by_family, ensure_ascii=False, sort_keys=True),
    }
    return summary, fold_df


def save_json(path: Path, payload: Dict) -> None:
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def finalize_submission(sub_df: pd.DataFrame, out_name: str) -> None:
    sub_df = sub_df.copy()
    sub_df = sub_df.rename(
        columns={
            DATE_COL: "Date",
            TARGET_REVENUE: "Revenue",
            TARGET_COGS: "COGS",
        }
    )

    if SAMPLE_SUBMISSION_FILE.exists():
        sample_sub = pd.read_csv(SAMPLE_SUBMISSION_FILE)
        sample_sub["Date"] = pd.to_datetime(sample_sub["Date"])
        sub_df["Date"] = pd.to_datetime(sub_df["Date"])
        sub_df = sample_sub[["Date"]].merge(sub_df, on="Date", how="left")

    sub_df["Revenue"] = pd.to_numeric(sub_df["Revenue"], errors="coerce").clip(lower=0)
    sub_df["COGS"] = pd.to_numeric(sub_df["COGS"], errors="coerce").clip(lower=0)

    if sub_df[["Revenue", "COGS"]].isna().any().any():
        raise ValueError(f"Submission {out_name} còn NaN. Hãy kiểm tra lại prediction.")

    sub_df["Date"] = pd.to_datetime(sub_df["Date"]).dt.strftime("%Y-%m-%d")
    sub_df.to_csv(OUTPUT_DIR / out_name, index=False)




In [9]:
# ============================================================
# MAIN
# ============================================================
def main() -> None:
    seed_everything(RANDOM_SEED)

    if not DATA_FILE.exists():
        raise FileNotFoundError(
            f"Không tìm thấy {DATA_FILE}. Hãy chạy build_daily_model_table.py trước."
        )

    print(f"DATA_FILE: {DATA_FILE}")
    print(f"MAX_CPU_THREADS: {MAX_CPU_THREADS}")
    print(f"Quick tune config: N_FOLDS={N_FOLDS}, HORIZON={HORIZON}")
    print(f"Search iters: LGBM={LGBM_N_ITER}, XGB={XGB_N_ITER}, CAT={CAT_N_ITER}")

    df = pd.read_csv(DATA_FILE, parse_dates=[DATE_COL])
    df.columns = [c.strip().lower() for c in df.columns]
    df = df.sort_values(DATE_COL).reset_index(drop=True)

    train_raw = df[df[TARGET_REVENUE].notna()].copy()
    future_raw = df[df[TARGET_REVENUE].isna()].copy()

    calendar_cols = [
        "day",
        "day_of_week",
        "day_of_year",
        "week_of_year",
        "month",
        "quarter",
        "year",
        "is_weekend",
        "is_month_start",
        "is_month_end",
        "is_quarter_start",
        "is_quarter_end",
        "dow_sin",
        "dow_cos",
        "month_sin",
        "month_cos",
        "doy_sin",
        "doy_cos",
    ]
    calendar_cols = [c for c in calendar_cols if c in df.columns]

    known_future_cols = [
        # trend
        "time_idx",
        "time_idx_sq",

        # holiday / mega sale
        "is_holiday_fixed",
        "is_mega_sale_0909",
        "is_mega_sale_1111",
        "is_mega_sale_1212",
        "is_mega_sale",
        "days_to_next_mega_sale",
        "days_since_prev_mega_sale",
        "is_pre_mega_sale_3d",
        "is_post_mega_sale_3d",

        # promo schedule mới, bám theo EDA Outdoor / Streetwear
        "is_promo_active",
        "promo_active_cnt",
        "promo_global_cnt",
        "promo_targeted_cnt",
        "promo_global_ratio",
        "promo_discount_mean",
        "promo_global_discount_mean",
        "promo_targeted_discount_mean",
        "promo_intensity",
        "promo_pct_cnt",
        "promo_fixed_cnt",
        "promo_stackable_cnt",
        "promo_outdoor_direct_cnt",
        "promo_outdoor_direct_discount_mean",
        "promo_streetwear_direct_cnt",
        "promo_streetwear_direct_discount_mean",
        "days_to_next_promo_start",
        "days_since_prev_promo_end",
        "is_promo_start_day",
        "is_promo_end_day",
        "is_pre_promo_3d",
        "is_post_promo_3d",
        "is_pre_promo_7d",
        "is_post_promo_7d",
    ]
    known_future_cols = [c for c in known_future_cols if c in df.columns]

    revenue_ts_cols = [
        c for c in df.columns
        if c.startswith("revenue_lag_")
        or c.startswith("revenue_roll_")
        or c.startswith("revenue_ewm_")
    ]
    cogs_ts_cols = [
        c for c in df.columns
        if c.startswith("cogs_lag_")
        or c.startswith("cogs_roll_")
        or c.startswith("cogs_ewm_")
    ]

    revenue_feature_cols = calendar_cols + known_future_cols + revenue_ts_cols
    cogs_feature_cols = calendar_cols + known_future_cols + cogs_ts_cols

    # Không dùng current-day revenue/cogs làm feature để tránh leakage
    forbidden = {DATE_COL, TARGET_REVENUE, TARGET_COGS}
    revenue_feature_cols = [c for c in revenue_feature_cols if c not in forbidden]
    cogs_feature_cols = [c for c in cogs_feature_cols if c not in forbidden]

    print(f"Revenue features: {len(revenue_feature_cols)}")
    print(f"COGS features: {len(cogs_feature_cols)}")
    print(f"Known future features: {len(known_future_cols)}")

    train_rev_df = (
        train_raw[[DATE_COL, TARGET_REVENUE] + revenue_feature_cols]
        .dropna()
        .reset_index(drop=True)
    )
    train_cogs_df = (
        train_raw[[DATE_COL, TARGET_COGS] + cogs_feature_cols]
        .dropna()
        .reset_index(drop=True)
    )

    rev_splits = build_backtest_splits(train_rev_df, N_FOLDS, HORIZON, MIN_TRAIN_ROWS)
    cogs_splits = build_backtest_splits(train_cogs_df, N_FOLDS, HORIZON, MIN_TRAIN_ROWS)

    if not rev_splits or not cogs_splits:
        raise ValueError("Không tạo được backtest splits. Hãy giảm N_FOLDS/HORIZON hoặc MIN_TRAIN_ROWS.")

    runtime_choices = detect_runtime_choices()
    print("Runtime choices:")
    for fam, choice in runtime_choices.items():
        print(f"  - {fam}: requested={choice.requested}, actual={choice.actual}")

    family_iters = {
        "lgbm": LGBM_N_ITER,
        "xgb": XGB_N_ITER,
        "cat": CAT_N_ITER,
    }

    target_configs = [
        (TARGET_REVENUE, train_rev_df, revenue_feature_cols, rev_splits),
        (TARGET_COGS, train_cogs_df, cogs_feature_cols, cogs_splits),
    ]

    best_family_by_target: Dict[str, str] = {}
    best_params_by_target: Dict[str, Dict[str, Dict]] = {}
    compare_plus_by_target: Dict[str, pd.DataFrame] = {}

    for target_col, df_target, feature_cols, splits in target_configs:
        family_best_params: Dict[str, Dict] = {}
        family_compare_rows: List[Dict] = []

        for family in ["lgbm", "xgb", "cat"]:
            print("=" * 80)
            print(f"[{target_col}] Quick random search for {family}")

            trials_df, best_summary, best_folds_df = random_search_family(
                df_target=df_target,
                target_col=target_col,
                feature_cols=feature_cols,
                family=family,
                n_iter=family_iters[family],
                splits=splits,
                runtime_choices=runtime_choices,
                seed_offset={"lgbm": 100, "xgb": 200, "cat": 300}[family]
                + (0 if target_col == TARGET_REVENUE else 1000),
            )

            if SAVE_INTERMEDIATE:
                trials_path = OUTPUT_DIR / f"{target_col}_{family}_deep_trials.csv"
                folds_path = OUTPUT_DIR / f"{target_col}_{family}_best_folds.csv"
                best_json_path = OUTPUT_DIR / f"{target_col}_{family}_best.json"
                trials_df.to_csv(trials_path, index=False)
                best_folds_df.to_csv(folds_path, index=False)
                save_json(best_json_path, best_summary)

            params = json.loads(best_summary["params_json"])
            family_best_params[family] = params
            family_compare_rows.append(best_summary)

        compare_df = (
            pd.DataFrame(family_compare_rows)
            .sort_values(["mean_rmse", "mean_mae"])
            .reset_index(drop=True)
        )

        best_family = str(compare_df.iloc[0]["model_family"])
        best_family_by_target[target_col] = best_family
        best_params_by_target[target_col] = family_best_params

        print("-" * 80)
        print(f"[{target_col}] best single family: {best_family}")
        print(compare_df[["model_family", "mean_rmse", "mean_mae", "mean_r2", "runtime_used"]])

        ensemble_summary, ensemble_folds_df = evaluate_ensemble_best_models(
            df_target=df_target,
            target_col=target_col,
            feature_cols=feature_cols,
            best_params_by_family=family_best_params,
            splits=splits,
            runtime_choices=runtime_choices,
        )

        if SAVE_INTERMEDIATE:
            ensemble_folds_path = OUTPUT_DIR / f"{target_col}_ensemble_mean_folds.csv"
            ensemble_json_path = OUTPUT_DIR / f"{target_col}_ensemble_mean.json"
            ensemble_folds_df.to_csv(ensemble_folds_path, index=False)
            save_json(ensemble_json_path, ensemble_summary)

        compare_plus_df = pd.concat([compare_df, pd.DataFrame([ensemble_summary])], ignore_index=True)
        compare_plus_df = compare_plus_df.sort_values(["mean_rmse", "mean_mae"]).reset_index(drop=True)
        compare_plus_by_target[target_col] = compare_plus_df

        compare_plus_path = OUTPUT_DIR / f"{target_col}_family_compare_plus_ensemble.csv"
        compare_plus_df.to_csv(compare_plus_path, index=False)
        print(f"Saved compare+ensemble to: {compare_plus_path}")

    # ========================================================
    # FULL TRAIN + FUTURE FORECAST
    # ========================================================
    if len(future_raw) == 0:
        print("[WARN] Không có future rows. Bỏ qua bước forecast submission.")
        return

    future_dates_sorted = sorted(pd.to_datetime(future_raw[DATE_COL]).tolist())

    full_models: Dict[str, Dict[str, object]] = {TARGET_REVENUE: {}, TARGET_COGS: {}}

    for target_col, df_target, feature_cols, _ in target_configs:
        for family in ["lgbm", "xgb", "cat"]:
            params = best_params_by_target[target_col][family]
            model = fit_full_model(
                df_target=df_target,
                target_col=target_col,
                feature_cols=feature_cols,
                family=family,
                params=params,
                runtime_choices=runtime_choices,
            )
            full_models[target_col][family] = model
            if SAVE_MODELS:
                joblib.dump(model, MODEL_DIR / f"best_{target_col}_{family}.joblib")

    family_preds: Dict[str, Dict[str, pd.DataFrame]] = {TARGET_REVENUE: {}, TARGET_COGS: {}}
    for target_col, df_target, feature_cols, _ in target_configs:
        for family in ["lgbm", "xgb", "cat"]:
            pred_df = recursive_forecast(
                model=full_models[target_col][family],
                train_df=df_target[[DATE_COL, target_col] + feature_cols],
                future_dates=future_dates_sorted,
                target_col=target_col,
                feature_cols=feature_cols,
                known_future_df=future_raw[[DATE_COL] + feature_cols],
            )
            family_preds[target_col][family] = pred_df

            if SAVE_INTERMEDIATE:
                pred_out_path = OUTPUT_DIR / f"future_pred_{target_col}_{family}.csv"
                pred_df.to_csv(pred_out_path, index=False)

    revenue_ensemble = pd.DataFrame({DATE_COL: future_dates_sorted})
    cogs_ensemble = pd.DataFrame({DATE_COL: future_dates_sorted})

    revenue_ensemble[TARGET_REVENUE] = np.mean(
        np.vstack([
            family_preds[TARGET_REVENUE][fam][TARGET_REVENUE].values
            for fam in ["lgbm", "xgb", "cat"]
        ]),
        axis=0,
    )
    cogs_ensemble[TARGET_COGS] = np.mean(
        np.vstack([
            family_preds[TARGET_COGS][fam][TARGET_COGS].values
            for fam in ["lgbm", "xgb", "cat"]
        ]),
        axis=0,
    )

    if SAVE_INTERMEDIATE:
        revenue_ensemble.to_csv(OUTPUT_DIR / "future_pred_revenue_ensemble_mean.csv", index=False)
        cogs_ensemble.to_csv(OUTPUT_DIR / "future_pred_cogs_ensemble_mean.csv", index=False)

    best_rev_family = best_family_by_target[TARGET_REVENUE]
    best_cogs_family = best_family_by_target[TARGET_COGS]

    best_single_submission = pd.DataFrame({DATE_COL: future_dates_sorted})
    best_single_submission = best_single_submission.merge(
        family_preds[TARGET_REVENUE][best_rev_family][[DATE_COL, TARGET_REVENUE]],
        on=DATE_COL,
        how="left",
    )
    best_single_submission = best_single_submission.merge(
        family_preds[TARGET_COGS][best_cogs_family][[DATE_COL, TARGET_COGS]],
        on=DATE_COL,
        how="left",
    )

    ensemble_submission = pd.DataFrame({DATE_COL: future_dates_sorted})
    ensemble_submission = ensemble_submission.merge(revenue_ensemble, on=DATE_COL, how="left")
    ensemble_submission = ensemble_submission.merge(cogs_ensemble, on=DATE_COL, how="left")

    if SAVE_ALT_SUBMISSIONS:
        finalize_submission(best_single_submission, "submission_best_single_deep.csv")
        finalize_submission(ensemble_submission, "submission_ensemble_mean.csv")

    best_rev_final = str(compare_plus_by_target[TARGET_REVENUE].iloc[0]["model_family"])
    best_cogs_final = str(compare_plus_by_target[TARGET_COGS].iloc[0]["model_family"])

    final_submission = pd.DataFrame({DATE_COL: future_dates_sorted})

    if best_rev_final == "ensemble_mean":
        final_submission = final_submission.merge(revenue_ensemble, on=DATE_COL, how="left")
    else:
        final_submission = final_submission.merge(
            family_preds[TARGET_REVENUE][best_rev_final][[DATE_COL, TARGET_REVENUE]],
            on=DATE_COL,
            how="left",
        )

    if best_cogs_final == "ensemble_mean":
        final_submission = final_submission.merge(cogs_ensemble, on=DATE_COL, how="left")
    else:
        final_submission = final_submission.merge(
            family_preds[TARGET_COGS][best_cogs_final][[DATE_COL, TARGET_COGS]],
            on=DATE_COL,
            how="left",
        )

    finalize_submission(final_submission, "submission.csv")

# ============================================================
# MAIN
# ============================================================
def main() -> None:
    seed_everything(RANDOM_SEED)

    if not DATA_FILE.exists():
        raise FileNotFoundError(
            f"Không tìm thấy {DATA_FILE}. Hãy chạy build_daily_model_table.py trước."
        )

    print(f"DATA_FILE: {DATA_FILE}")
    print(f"MAX_CPU_THREADS: {MAX_CPU_THREADS}")
    print(f"Quick tune config: N_FOLDS={N_FOLDS}, HORIZON={HORIZON}")
    print(f"Search iters: LGBM={LGBM_N_ITER}, XGB={XGB_N_ITER}, CAT={CAT_N_ITER}")

    df = pd.read_csv(DATA_FILE, parse_dates=[DATE_COL])
    df.columns = [c.strip().lower() for c in df.columns]
    df = df.sort_values(DATE_COL).reset_index(drop=True)

    train_raw = df[df[TARGET_REVENUE].notna()].copy()
    future_raw = df[df[TARGET_REVENUE].isna()].copy()

    calendar_cols = [
        "day",
        "day_of_week",
        "day_of_year",
        "week_of_year",
        "month",
        "quarter",
        "year",
        "is_weekend",
        "is_month_start",
        "is_month_end",
        "is_quarter_start",
        "is_quarter_end",
        "dow_sin",
        "dow_cos",
        "month_sin",
        "month_cos",
        "doy_sin",
        "doy_cos",
    ]
    calendar_cols = [c for c in calendar_cols if c in df.columns]

    known_future_cols = [
        # trend
        "time_idx",
        "time_idx_sq",

        # holiday / mega sale
        "is_holiday_fixed",
        "is_mega_sale_0909",
        "is_mega_sale_1111",
        "is_mega_sale_1212",
        "is_mega_sale",
        "days_to_next_mega_sale",
        "days_since_prev_mega_sale",
        "is_pre_mega_sale_3d",
        "is_post_mega_sale_3d",

        # promo schedule mới, bám theo EDA Outdoor / Streetwear
        "is_promo_active",
        "promo_active_cnt",
        "promo_global_cnt",
        "promo_targeted_cnt",
        "promo_global_ratio",
        "promo_discount_mean",
        "promo_global_discount_mean",
        "promo_targeted_discount_mean",
        "promo_intensity",
        "promo_pct_cnt",
        "promo_fixed_cnt",
        "promo_stackable_cnt",
        "promo_outdoor_direct_cnt",
        "promo_outdoor_direct_discount_mean",
        "promo_streetwear_direct_cnt",
        "promo_streetwear_direct_discount_mean",
        "days_to_next_promo_start",
        "days_since_prev_promo_end",
        "is_promo_start_day",
        "is_promo_end_day",
        "is_pre_promo_3d",
        "is_post_promo_3d",
        "is_pre_promo_7d",
        "is_post_promo_7d",
    ]
    known_future_cols = [c for c in known_future_cols if c in df.columns]

    revenue_ts_cols = [
        c for c in df.columns
        if c.startswith("revenue_lag_")
        or c.startswith("revenue_roll_")
        or c.startswith("revenue_ewm_")
    ]
    cogs_ts_cols = [
        c for c in df.columns
        if c.startswith("cogs_lag_")
        or c.startswith("cogs_roll_")
        or c.startswith("cogs_ewm_")
    ]

    revenue_feature_cols = calendar_cols + known_future_cols + revenue_ts_cols
    cogs_feature_cols = calendar_cols + known_future_cols + cogs_ts_cols

    # Không dùng current-day revenue/cogs làm feature để tránh leakage
    forbidden = {DATE_COL, TARGET_REVENUE, TARGET_COGS}
    revenue_feature_cols = [c for c in revenue_feature_cols if c not in forbidden]
    cogs_feature_cols = [c for c in cogs_feature_cols if c not in forbidden]

    print(f"Revenue features: {len(revenue_feature_cols)}")
    print(f"COGS features: {len(cogs_feature_cols)}")
    print(f"Known future features: {len(known_future_cols)}")

    train_rev_df = (
        train_raw[[DATE_COL, TARGET_REVENUE] + revenue_feature_cols]
        .dropna()
        .reset_index(drop=True)
    )
    train_cogs_df = (
        train_raw[[DATE_COL, TARGET_COGS] + cogs_feature_cols]
        .dropna()
        .reset_index(drop=True)
    )

    rev_splits = build_backtest_splits(train_rev_df, N_FOLDS, HORIZON, MIN_TRAIN_ROWS)
    cogs_splits = build_backtest_splits(train_cogs_df, N_FOLDS, HORIZON, MIN_TRAIN_ROWS)

    if not rev_splits or not cogs_splits:
        raise ValueError("Không tạo được backtest splits. Hãy giảm N_FOLDS/HORIZON hoặc MIN_TRAIN_ROWS.")

    runtime_choices = detect_runtime_choices()
    print("Runtime choices:")
    for fam, choice in runtime_choices.items():
        print(f"  - {fam}: requested={choice.requested}, actual={choice.actual}")

    family_iters = {
        "lgbm": LGBM_N_ITER,
        "xgb": XGB_N_ITER,
        "cat": CAT_N_ITER,
    }

    target_configs = [
        (TARGET_REVENUE, train_rev_df, revenue_feature_cols, rev_splits),
        (TARGET_COGS, train_cogs_df, cogs_feature_cols, cogs_splits),
    ]

    best_family_by_target: Dict[str, str] = {}
    best_params_by_target: Dict[str, Dict[str, Dict]] = {}
    compare_plus_by_target: Dict[str, pd.DataFrame] = {}

    for target_col, df_target, feature_cols, splits in target_configs:
        family_best_params: Dict[str, Dict] = {}
        family_compare_rows: List[Dict] = []

        for family in ["lgbm", "xgb", "cat"]:
            print("=" * 80)
            print(f"[{target_col}] Quick random search for {family}")

            trials_df, best_summary, best_folds_df = random_search_family(
                df_target=df_target,
                target_col=target_col,
                feature_cols=feature_cols,
                family=family,
                n_iter=family_iters[family],
                splits=splits,
                runtime_choices=runtime_choices,
                seed_offset={"lgbm": 100, "xgb": 200, "cat": 300}[family]
                + (0 if target_col == TARGET_REVENUE else 1000),
            )

            if SAVE_INTERMEDIATE:
                trials_path = OUTPUT_DIR / f"{target_col}_{family}_deep_trials.csv"
                folds_path = OUTPUT_DIR / f"{target_col}_{family}_best_folds.csv"
                best_json_path = OUTPUT_DIR / f"{target_col}_{family}_best.json"
                trials_df.to_csv(trials_path, index=False)
                best_folds_df.to_csv(folds_path, index=False)
                save_json(best_json_path, best_summary)

            params = json.loads(best_summary["params_json"])
            family_best_params[family] = params
            family_compare_rows.append(best_summary)

        compare_df = (
            pd.DataFrame(family_compare_rows)
            .sort_values(["mean_rmse", "mean_mae"])
            .reset_index(drop=True)
        )

        best_family = str(compare_df.iloc[0]["model_family"])
        best_family_by_target[target_col] = best_family
        best_params_by_target[target_col] = family_best_params

        print("-" * 80)
        print(f"[{target_col}] best single family: {best_family}")
        print(compare_df[["model_family", "mean_rmse", "mean_mae", "mean_r2", "runtime_used"]])

        ensemble_summary, ensemble_folds_df = evaluate_ensemble_best_models(
            df_target=df_target,
            target_col=target_col,
            feature_cols=feature_cols,
            best_params_by_family=family_best_params,
            splits=splits,
            runtime_choices=runtime_choices,
        )

        if SAVE_INTERMEDIATE:
            ensemble_folds_path = OUTPUT_DIR / f"{target_col}_ensemble_mean_folds.csv"
            ensemble_json_path = OUTPUT_DIR / f"{target_col}_ensemble_mean.json"
            ensemble_folds_df.to_csv(ensemble_folds_path, index=False)
            save_json(ensemble_json_path, ensemble_summary)

        compare_plus_df = pd.concat([compare_df, pd.DataFrame([ensemble_summary])], ignore_index=True)
        compare_plus_df = compare_plus_df.sort_values(["mean_rmse", "mean_mae"]).reset_index(drop=True)
        compare_plus_by_target[target_col] = compare_plus_df

        compare_plus_path = OUTPUT_DIR / f"{target_col}_family_compare_plus_ensemble.csv"
        compare_plus_df.to_csv(compare_plus_path, index=False)
        print(f"Saved compare+ensemble to: {compare_plus_path}")

    # ========================================================
    # FULL TRAIN + FUTURE FORECAST
    # ========================================================
    if len(future_raw) == 0:
        print("[WARN] Không có future rows. Bỏ qua bước forecast submission.")
        return

    future_dates_sorted = sorted(pd.to_datetime(future_raw[DATE_COL]).tolist())

    full_models: Dict[str, Dict[str, object]] = {TARGET_REVENUE: {}, TARGET_COGS: {}}

    for target_col, df_target, feature_cols, _ in target_configs:
        for family in ["lgbm", "xgb", "cat"]:
            params = best_params_by_target[target_col][family]
            model = fit_full_model(
                df_target=df_target,
                target_col=target_col,
                feature_cols=feature_cols,
                family=family,
                params=params,
                runtime_choices=runtime_choices,
            )
            full_models[target_col][family] = model
            if SAVE_MODELS:
                joblib.dump(model, MODEL_DIR / f"best_{target_col}_{family}.joblib")

    family_preds: Dict[str, Dict[str, pd.DataFrame]] = {TARGET_REVENUE: {}, TARGET_COGS: {}}
    for target_col, df_target, feature_cols, _ in target_configs:
        for family in ["lgbm", "xgb", "cat"]:
            pred_df = recursive_forecast(
                model=full_models[target_col][family],
                train_df=df_target[[DATE_COL, target_col] + feature_cols],
                future_dates=future_dates_sorted,
                target_col=target_col,
                feature_cols=feature_cols,
                known_future_df=future_raw[[DATE_COL] + feature_cols],
            )
            family_preds[target_col][family] = pred_df

            if SAVE_INTERMEDIATE:
                pred_out_path = OUTPUT_DIR / f"future_pred_{target_col}_{family}.csv"
                pred_df.to_csv(pred_out_path, index=False)

    revenue_ensemble = pd.DataFrame({DATE_COL: future_dates_sorted})
    cogs_ensemble = pd.DataFrame({DATE_COL: future_dates_sorted})

    revenue_ensemble[TARGET_REVENUE] = np.mean(
        np.vstack([
            family_preds[TARGET_REVENUE][fam][TARGET_REVENUE].values
            for fam in ["lgbm", "xgb", "cat"]
        ]),
        axis=0,
    )
    cogs_ensemble[TARGET_COGS] = np.mean(
        np.vstack([
            family_preds[TARGET_COGS][fam][TARGET_COGS].values
            for fam in ["lgbm", "xgb", "cat"]
        ]),
        axis=0,
    )

    if SAVE_INTERMEDIATE:
        revenue_ensemble.to_csv(OUTPUT_DIR / "future_pred_revenue_ensemble_mean.csv", index=False)
        cogs_ensemble.to_csv(OUTPUT_DIR / "future_pred_cogs_ensemble_mean.csv", index=False)

    best_rev_family = best_family_by_target[TARGET_REVENUE]
    best_cogs_family = best_family_by_target[TARGET_COGS]

    best_single_submission = pd.DataFrame({DATE_COL: future_dates_sorted})
    best_single_submission = best_single_submission.merge(
        family_preds[TARGET_REVENUE][best_rev_family][[DATE_COL, TARGET_REVENUE]],
        on=DATE_COL,
        how="left",
    )
    best_single_submission = best_single_submission.merge(
        family_preds[TARGET_COGS][best_cogs_family][[DATE_COL, TARGET_COGS]],
        on=DATE_COL,
        how="left",
    )

    ensemble_submission = pd.DataFrame({DATE_COL: future_dates_sorted})
    ensemble_submission = ensemble_submission.merge(revenue_ensemble, on=DATE_COL, how="left")
    ensemble_submission = ensemble_submission.merge(cogs_ensemble, on=DATE_COL, how="left")

    if SAVE_ALT_SUBMISSIONS:
        finalize_submission(best_single_submission, "submission_best_single_deep.csv")
        finalize_submission(ensemble_submission, "submission_ensemble_mean.csv")

    best_rev_final = str(compare_plus_by_target[TARGET_REVENUE].iloc[0]["model_family"])
    best_cogs_final = str(compare_plus_by_target[TARGET_COGS].iloc[0]["model_family"])

    final_submission = pd.DataFrame({DATE_COL: future_dates_sorted})

    if best_rev_final == "ensemble_mean":
        final_submission = final_submission.merge(revenue_ensemble, on=DATE_COL, how="left")
    else:
        final_submission = final_submission.merge(
            family_preds[TARGET_REVENUE][best_rev_final][[DATE_COL, TARGET_REVENUE]],
            on=DATE_COL,
            how="left",
        )

    if best_cogs_final == "ensemble_mean":
        final_submission = final_submission.merge(cogs_ensemble, on=DATE_COL, how="left")
    else:
        final_submission = final_submission.merge(
            family_preds[TARGET_COGS][best_cogs_final][[DATE_COL, TARGET_COGS]],
            on=DATE_COL,
            how="left",
        )

    finalize_submission(final_submission, "submission.csv")

    # ============================================================
    # REPORT ARTIFACTS: FEATURE IMPORTANCE
    # ============================================================

    def get_feature_importance(model, feature_cols):
        if hasattr(model, "feature_importances_"):
            importance = model.feature_importances_
        elif hasattr(model, "get_feature_importance"):
            importance = model.get_feature_importance()
        else:
            return None

        return (
            pd.DataFrame({
                "feature": feature_cols,
                "importance": importance
            })
            .sort_values("importance", ascending=False)
            .reset_index(drop=True)
        )


    for target_col, feature_cols in [
        (TARGET_REVENUE, revenue_feature_cols),
        (TARGET_COGS, cogs_feature_cols)
    ]:
        best_family = str(compare_plus_by_target[target_col].iloc[0]["model_family"])

        if best_family == "ensemble_mean":
            single_compare = compare_plus_by_target[target_col]
            single_compare = single_compare[single_compare["model_family"] != "ensemble_mean"]
            explain_family = str(single_compare.iloc[0]["model_family"])
        else:
            explain_family = best_family

        model = full_models[target_col][explain_family]

        fi = get_feature_importance(model, feature_cols)
        if fi is None:
            print(f"[WARN] Không lấy được feature importance cho {target_col} - {explain_family}")
            continue

        fi_path = OUTPUT_DIR / f"feature_importance_{target_col}_{explain_family}.csv"
        fi.to_csv(fi_path, index=False)

        top_fi = fi.head(20).sort_values("importance", ascending=True)

        plt.figure(figsize=(8, 6))
        plt.barh(top_fi["feature"], top_fi["importance"])
        plt.title(f"Top 20 Feature Importance - {target_col.upper()} ({explain_family})")
        plt.xlabel("Importance")
        plt.ylabel("Feature")
        plt.tight_layout()

        fig_path = OUTPUT_DIR / f"feature_importance_{target_col}_{explain_family}.png"
        plt.savefig(fig_path, dpi=300, bbox_inches="tight")
        plt.close()

        print(f"Saved feature importance for {target_col}: {fig_path}")

        final_manifest = {
            "best_single_family": {
                TARGET_REVENUE: best_rev_family,
                TARGET_COGS: best_cogs_family,
            },
            "best_final_choice": {
                TARGET_REVENUE: best_rev_final,
                TARGET_COGS: best_cogs_final,
            },
            "cpu_threads": MAX_CPU_THREADS,
            "runtime_choices": {k: v.actual for k, v in runtime_choices.items()},
            "search_iters": {
                "lgbm": LGBM_N_ITER,
                "xgb": XGB_N_ITER,
                "cat": CAT_N_ITER,
            },
            "folds": N_FOLDS,
            "horizon": HORIZON,
            "min_train_rows": MIN_TRAIN_ROWS,
            "save_intermediate": SAVE_INTERMEDIATE,
            "save_models": SAVE_MODELS,
            "save_alt_submissions": SAVE_ALT_SUBMISSIONS,
            "data_file": str(DATA_FILE),
            "revenue_features": len(revenue_feature_cols),
            "cogs_features": len(cogs_feature_cols),
            "known_future_features": len(known_future_cols),
        }
        save_json(OUTPUT_DIR / "deep_tune_run_manifest.json", final_manifest)

        print("Saved final files:")
        print(" -", OUTPUT_DIR / "submission.csv")
        print(" -", OUTPUT_DIR / "revenue_family_compare_plus_ensemble.csv")
        print(" -", OUTPUT_DIR / "cogs_family_compare_plus_ensemble.csv")
        print(" -", OUTPUT_DIR / "deep_tune_run_manifest.json")
        if SAVE_MODELS:
            print("Saved models to:", MODEL_DIR)
        print("Done tune_model_section.py!")


if __name__ == "__main__":
    main()

DATA_FILE: D:\CODEPY\Fashion_Ecom_Sales_Forecasting\outputs\daily_features_tuned_best.csv
MAX_CPU_THREADS: 12
Quick tune config: N_FOLDS=2, HORIZON=60
Search iters: LGBM=12, XGB=12, CAT=12
Revenue features: 82
COGS features: 82
Known future features: 35
Runtime choices:
  - lgbm: requested=gpu, actual=gpu
  - xgb: requested=gpu, actual=gpu
  - cat: requested=gpu, actual=gpu
[revenue] Quick random search for lgbm
[revenue] lgbm trial 1/12 | RMSE=593874.108 | MAE=468107.548 | R2=0.2888
[revenue] lgbm trial 2/12 | RMSE=617582.776 | MAE=490008.716 | R2=0.2423
[revenue] lgbm trial 3/12 | RMSE=659624.203 | MAE=528280.971 | R2=0.1479
[revenue] lgbm trial 4/12 | RMSE=590907.127 | MAE=472697.703 | R2=0.3134
[revenue] lgbm trial 5/12 | RMSE=615436.553 | MAE=490343.851 | R2=0.2520
[revenue] lgbm trial 6/12 | RMSE=610026.863 | MAE=480591.241 | R2=0.2551
[revenue] lgbm trial 7/12 | RMSE=612388.348 | MAE=492600.746 | R2=0.2418
[revenue] lgbm trial 8/12 | RMSE=626729.672 | MAE=501331.067 | R2=0.2263
